In [0]:

# ============================================================
# SECTION A — CONFIGURATION
# CELL 1 — IMPORTS, PATHS, AND PRODUCTS SCHEMA
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType,
    DoubleType
)

from datetime import datetime, timezone
import uuid

# Dataset configuration
DATASET_NAME = "products"
SOURCE_SYSTEM = "product_api"

# ADLS Gen2 configuration
STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"

BASE_PATH = (
    f"abfss://{CONTAINER}@"
    f"{STORAGE_ACCOUNT}.dfs.core.windows.net/"
)

LANDING_PATH = f"{BASE_PATH}landing/{DATASET_NAME}/"
BRONZE_PATH = f"{BASE_PATH}bronze/{DATASET_NAME}/"
MANIFEST_PATH = f"{BASE_PATH}metadata/file_manifest/"

# Explicit source schema
PRODUCTS_SCHEMA = StructType([
    StructField("product_id", LongType(), nullable=False),
    StructField("product_name", StringType(), nullable=True),
    StructField("category", StringType(), nullable=True),
    StructField("brand", StringType(), nullable=True),
    StructField("price", DoubleType(), nullable=True)
])

# Bronze audit columns
BRONZE_METADATA_COLUMNS = [
    "_source_file",
    "_source_path",
    "_dataset",
    "_batch_id",
    "_run_id",
    "_ingested_at",
    "_source_system"
]

print("Dataset:", DATASET_NAME)
print("Source system:", SOURCE_SYSTEM)
print("Landing path:", LANDING_PATH)
print("Bronze path:", BRONZE_PATH)
print("Manifest path:", MANIFEST_PATH)
print("Products schema fields:", len(PRODUCTS_SCHEMA.fields))


Dataset: products
Source system: product_api
Landing path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/
Bronze path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/
Manifest path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/file_manifest/
Products schema fields: 5


In [0]:

# ============================================================
# SECTION A — CONFIGURATION
# CELL 2 — RUN-MODE WIDGETS
# ============================================================

# Create widgets only if they don't already exist.
# Remove existing widgets first so their definitions are refreshed.
for widget_name in [
    "run_mode",
    "backfill_start_date",
    "backfill_end_date",
    "reprocess_successful",
    "manifest_run_id"
]:
    try:
        dbutils.widgets.remove(widget_name)
    except Exception:
        pass

dbutils.widgets.dropdown(
    "run_mode",
    "INCREMENTAL",
    ["INCREMENTAL", "BACKFILL", "FULL_LOAD"],
    "Run Mode"
)

dbutils.widgets.text(
    "backfill_start_date",
    "",
    "Backfill Start Date (YYYY-MM-DD)"
)

dbutils.widgets.text(
    "backfill_end_date",
    "",
    "Backfill End Date (YYYY-MM-DD)"
)

dbutils.widgets.dropdown(
    "reprocess_successful",
    "false",
    ["false", "true"],
    "Reprocess Successful Files?"
)

dbutils.widgets.text(
    "manifest_run_id",
    "",
    "Manifest Run ID (optional)"
)

RUN_MODE = dbutils.widgets.get("run_mode").strip().upper()
BACKFILL_START_DATE = dbutils.widgets.get(
    "backfill_start_date"
).strip()
BACKFILL_END_DATE = dbutils.widgets.get(
    "backfill_end_date"
).strip()
REPROCESS_SUCCESSFUL = (
    dbutils.widgets.get("reprocess_successful").strip().lower()
    == "true"
)
MANIFEST_RUN_ID = dbutils.widgets.get(
    "manifest_run_id"
).strip()

print("Run mode:", RUN_MODE)
print("Backfill start:", BACKFILL_START_DATE or "(not set)")
print("Backfill end:", BACKFILL_END_DATE or "(not set)")
print("Reprocess successful:", REPROCESS_SUCCESSFUL)
print("Manifest run ID supplied:", bool(MANIFEST_RUN_ID))


Run mode: FULL_LOAD
Backfill start: 2026-10-07
Backfill end: 2026-10-07
Reprocess successful: False
Manifest run ID supplied: False


In [0]:

# ============================================================
# SECTION A — CONFIGURATION
# CELL 3 — VALIDATE RUN-MODE PARAMETERS
# ============================================================

from datetime import datetime

VALID_RUN_MODES = {"INCREMENTAL", "BACKFILL", "FULL_LOAD"}

assert RUN_MODE in VALID_RUN_MODES, (
    f"Invalid run mode: {RUN_MODE}"
)

if RUN_MODE == "BACKFILL":
    assert BACKFILL_START_DATE, (
        "Backfill start date is required."
    )
    assert BACKFILL_END_DATE, (
        "Backfill end date is required."
    )

    try:
        start_date = datetime.strptime(
            BACKFILL_START_DATE, "%Y-%m-%d"
        ).date()

        end_date = datetime.strptime(
            BACKFILL_END_DATE, "%Y-%m-%d"
        ).date()
    except ValueError:
        raise ValueError(
            "Backfill dates must use YYYY-MM-DD format."
        )

    assert start_date <= end_date, (
        "Backfill start date must be on or before end date."
    )

print("PASS: Run mode is valid.")

if RUN_MODE == "BACKFILL":
    print("Backfill range:", start_date, "to", end_date)
else:
    print("Backfill date validation: not required")


PASS: Run mode is valid.
Backfill date validation: not required


In [0]:

# ============================================================
# SECTION B — DISCOVERY
# CELL 4 — RECURSIVE LANDING-FILE DISCOVERY
# Read-only; no manifest or Bronze writes
# ============================================================

def list_files_recursive(directory_path):
    discovered_files = []

    for entry in dbutils.fs.ls(directory_path):
        if entry.isDir():
            discovered_files.extend(
                list_files_recursive(entry.path)
            )
        else:
            discovered_files.append({
                "file_name": entry.name,
                "file_path": entry.path,
                "file_size_bytes": entry.size
            })

    return discovered_files


landing_files = list_files_recursive(LANDING_PATH)

landing_file_schema = StructType([
    StructField("file_name", StringType(), False),
    StructField("file_path", StringType(), False),
    StructField("file_size_bytes", LongType(), False)
])

if landing_files:
    landing_files_df = spark.createDataFrame(
        landing_files,
        schema=landing_file_schema
    )
else:
    landing_files_df = spark.createDataFrame(
        [],
        schema=landing_file_schema
    )

print("Landing files discovered:", landing_files_df.count())

display(
    landing_files_df.orderBy("file_name")
)


Landing files discovered: 2


file_name,file_path,file_size_bytes
products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,3662500
products_20261008.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=08/products_20261008.json,2615396


In [0]:

# ============================================================
# SECTION B — DISCOVERY
# CELL 5 — VALIDATE FILENAMES AND DERIVE DATA DATES
# Read-only
# ============================================================

from pyspark.sql import functions as F

dated_files_df = (
    landing_files_df
    .withColumn(
        "date_string",
        F.regexp_extract(
            F.col("file_name"),
            r"^products_(\d{8})\.json$",
            1
        )
    )
    .withColumn(
        "data_date",
        F.to_date("date_string", "yyyyMMdd")
    )
    .withColumn(
        "filename_status",
        F.when(
            F.col("date_string") == "",
            "INVALID_FILENAME"
        ).when(
            F.col("data_date").isNull(),
            "INVALID_DATE"
        ).otherwise("VALID")
    )
)

display(
    dated_files_df.select(
        "file_name",
        "file_size_bytes",
        "data_date",
        "filename_status"
    ).orderBy("file_name")
)

invalid_filename_count = dated_files_df.filter(
    F.col("filename_status") != "VALID"
).count()

print("Invalid filenames or dates:", invalid_filename_count)

assert invalid_filename_count == 0, (
    "Invalid landing filenames or dates found. "
    "Inspect the results before proceeding."
)

print("PASS: Landing filenames and dates validated")


file_name,file_size_bytes,data_date,filename_status
products_20261007.json,3662500,2026-10-07,VALID
products_20261008.json,2615396,2026-10-08,VALID


Invalid filenames or dates: 0
PASS: Landing filenames and dates validated


In [0]:

# ============================================================
# SECTION B — DISCOVERY
# CELL 6 — CALCULATE LANDING FILE HASHES
# Read-only; no manifest or Bronze writes
# ============================================================

from pyspark.sql import functions as F

binary_files_df = (
    spark.read
    .format("binaryFile")
    .load(LANDING_PATH + "**")
    .select(
        F.col("path").alias("file_path"),
        F.col("length").alias("binary_file_size_bytes"),
        F.sha2(F.col("content"), 256).alias("actual_content_hash")
    )
)

hashed_files_df = (
    dated_files_df
    .filter(F.col("filename_status") == "VALID")
    .join(
        binary_files_df,
        on="file_path",
        how="left"
    )
)

# Confirm every discovered file received a hash
missing_hash_count = hashed_files_df.filter(
    F.col("actual_content_hash").isNull()
).count()

display(
    hashed_files_df.select(
        "file_name",
        "data_date",
        "file_size_bytes",
        "actual_content_hash"
    ).orderBy("file_name")
)

print("Files discovered:", dated_files_df.count())
print("Files with missing hashes:", missing_hash_count)

assert missing_hash_count == 0, (
    "Some landing files could not be hashed."
)

print("PASS: Landing file hashes calculated")


file_name,data_date,file_size_bytes,actual_content_hash
products_20261007.json,2026-10-07,3662500,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118
products_20261008.json,2026-10-08,2615396,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b


Files discovered: 2
Files with missing hashes: 0
PASS: Landing file hashes calculated


In [0]:

# ============================================================
# SECTION B — DISCOVERY
# CELL 7 — RECONCILE LANDING FILES WITH MANIFEST
# Read-only; no manifest writes
# ============================================================

from pyspark.sql import functions as F

manifest_df = (
    spark.read.format("delta").load(MANIFEST_PATH)
    .filter(F.col("dataset") == DATASET_NAME)
    .select(
        "dataset",
        "file_name",
        "status",
        "content_hash",
        "data_date",
        "retry_count",
        "run_id",
        "lease_expires_at"
    )
)

# Fail safely if the real manifest contains duplicate file keys
duplicate_manifest_keys = (
    manifest_df
    .groupBy("dataset", "file_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_manifest_keys == 0, (
    "Duplicate Products file keys exist in the manifest. "
    "Resolve them before processing."
)

reconciliation_df = (
    hashed_files_df.alias("landing")
    .join(
        manifest_df.alias("manifest"),
        (
            F.col("landing.file_name")
            == F.col("manifest.file_name")
        ),
        "left"
    )
    .select(
        F.col("landing.file_name"),
        F.col("landing.file_path"),
        F.col("landing.file_size_bytes"),
        F.col("landing.data_date").alias("landing_data_date"),
        F.col("landing.actual_content_hash"),
        F.col("manifest.status").alias("manifest_status"),
        F.col("manifest.content_hash").alias("manifest_content_hash"),
        F.col("manifest.data_date").alias("manifest_data_date"),
        F.col("manifest.retry_count"),
        F.col("manifest.run_id"),
        F.col("manifest.lease_expires_at")
    )
    .withColumn(
        "discovery_status",
        F.when(
            F.col("manifest_status").isNull(),
            "UNREGISTERED"
        ).when(
            F.col("manifest_content_hash").isNull(),
            "MISSING_MANIFEST_HASH"
        ).when(
            F.col("actual_content_hash")
            != F.col("manifest_content_hash"),
            "CONTENT_CHANGED"
        ).otherwise("REGISTERED_MATCH")
    )
)

display(reconciliation_df.orderBy("file_name"))

print("Landing files:", hashed_files_df.count())
print("Unregistered files:", reconciliation_df.filter(
    F.col("discovery_status") == "UNREGISTERED"
).count())
print("Changed files:", reconciliation_df.filter(
    F.col("discovery_status") == "CONTENT_CHANGED"
).count())
print("Hash matches:", reconciliation_df.filter(
    F.col("discovery_status") == "REGISTERED_MATCH"
).count())


file_name,file_path,file_size_bytes,landing_data_date,actual_content_hash,manifest_status,manifest_content_hash,manifest_data_date,retry_count,run_id,lease_expires_at,discovery_status
products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,3662500,2026-10-07,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,SUCCESS,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,2026-10-07,0,2308384d-2c39-41e2-816a-4bea198b310a,null,REGISTERED_MATCH
products_20261008.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=08/products_20261008.json,2615396,2026-10-08,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b,SUCCESS,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b,2026-10-08,0,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:37:52.030Z,REGISTERED_MATCH


Landing files: 2
Unregistered files: 0
Changed files: 0
Hash matches: 2


In [0]:

# CELL 8 — RUN-MODE SELECTOR (READ-ONLY)

selector_df = (
    reconciliation_df
    .withColumn(
        "effective_data_date",
        F.coalesce(
            F.col("manifest_data_date"),
            F.col("landing_data_date")
        )
    )
    .withColumn(
        "selection_reason",
        F.when(
            F.col("discovery_status") == "UNREGISTERED",
            "REGISTER_REQUIRED"
        )
        .when(
            F.col("discovery_status") == "MISSING_MANIFEST_HASH",
            "REVIEW_REQUIRED"
        )
        .when(
            F.col("discovery_status") == "CONTENT_CHANGED",
            "CONTENT_CHANGED"
        )
        .when(
            F.col("manifest_status") == "PROCESSING",
            "SKIP_PROCESSING"
        )
        .when(
            F.col("manifest_status").isin("NEW", "FAILED"),
            "ELIGIBLE"
        )
        .when(
            F.col("manifest_status") == "SUCCESS",
            "SUCCESS_REPROCESS_CANDIDATE"
        )
        .otherwise("REVIEW_REQUIRED")
    )
)

if RUN_MODE == "INCREMENTAL":
    selected_candidates_df = selector_df.filter(
        F.col("selection_reason").isin(
            "REGISTER_REQUIRED",
            "ELIGIBLE",
            "CONTENT_CHANGED"
        )
    )

elif RUN_MODE == "BACKFILL":
    if not BACKFILL_START_DATE or not BACKFILL_END_DATE:
        raise ValueError(
            "BACKFILL requires both start and end dates."
        )

    start_date = F.to_date(
        F.lit(BACKFILL_START_DATE), "yyyy-MM-dd"
    )
    end_date = F.to_date(
        F.lit(BACKFILL_END_DATE), "yyyy-MM-dd"
    )

    backfill_reasons = [
        "REGISTER_REQUIRED",
        "ELIGIBLE",
        "CONTENT_CHANGED"
    ]

    if REPROCESS_SUCCESSFUL:
        backfill_reasons.append("SUCCESS_REPROCESS_CANDIDATE")

    selected_candidates_df = selector_df.filter(
        F.col("effective_data_date").between(start_date, end_date)
        & F.col("selection_reason").isin(backfill_reasons)
    )

elif RUN_MODE == "FULL_LOAD":
    # Full load explicitly includes successful files.
    # Processing files and review-required files remain excluded.
    selected_candidates_df = selector_df.filter(
        F.col("selection_reason").isin(
            "REGISTER_REQUIRED",
            "ELIGIBLE",
            "CONTENT_CHANGED",
            "SUCCESS_REPROCESS_CANDIDATE"
        )
    )

else:
    raise ValueError(f"Unsupported run mode: {RUN_MODE}")

print("Run mode:", RUN_MODE)
print("Reprocess successful:", REPROCESS_SUCCESSFUL)
print("Selected candidates:", selected_candidates_df.count())

display(
    selected_candidates_df.select(
        "file_name",
        "manifest_status",
        "discovery_status",
        "effective_data_date",
        "selection_reason"
    )
)


Run mode: FULL_LOAD
Reprocess successful: False
Selected candidates: 2


file_name,manifest_status,discovery_status,effective_data_date,selection_reason
products_20261007.json,SUCCESS,REGISTERED_MATCH,2026-10-07,SUCCESS_REPROCESS_CANDIDATE
products_20261008.json,SUCCESS,REGISTERED_MATCH,2026-10-08,SUCCESS_REPROCESS_CANDIDATE


In [0]:

# ============================================================
# SECTION B — DISCOVERY
# CELL 9 — SELECTION SAFETY GATE
# Read-only
# ============================================================

from pyspark.sql import functions as F

selected_count = selected_candidates_df.count()

duplicate_count = (
    selected_candidates_df
    .groupBy("file_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_hash_count = selected_candidates_df.filter(
    F.col("actual_content_hash").isNull()
    | (F.length("actual_content_hash") != 64)
).count()

invalid_date_count = selected_candidates_df.filter(
    F.col("effective_data_date").isNull()
).count()

processing_count = selected_candidates_df.filter(
    F.col("manifest_status") == "PROCESSING"
).count()

review_count = selected_candidates_df.filter(
    F.col("selection_reason") == "REVIEW_REQUIRED"
).count()

print("Selected files:", selected_count)
print("Duplicate file keys:", duplicate_count)
print("Invalid hashes:", invalid_hash_count)
print("Invalid dates:", invalid_date_count)
print("PROCESSING files selected:", processing_count)
print("Review-required files selected:", review_count)

assert duplicate_count == 0, "Duplicate file keys detected"
assert invalid_hash_count == 0, "Invalid file hashes detected"
assert invalid_date_count == 0, "Invalid file dates detected"
assert processing_count == 0, "PROCESSING file selected unexpectedly"
assert review_count == 0, "Review-required file selected unexpectedly"

print("PASS: Selection safety gate")


Selected files: 1
Duplicate file keys: 0
Invalid hashes: 0
Invalid dates: 0
PROCESSING files selected: 0
Review-required files selected: 0
PASS: Selection safety gate


In [0]:

# ============================================================
# SECTION C — MANIFEST MANAGEMENT
# CELL 10 — PREPARE NEW REGISTRATION RECORDS
# Read-only
# ============================================================

from pyspark.sql import functions as F

registration_candidates_df = (
    selector_df
    .filter(
        (F.col("selection_reason") == "REGISTER_REQUIRED")
        & (F.col("discovery_status") == "UNREGISTERED")
    )
    .select(
        F.lit(DATASET_NAME).alias("dataset"),
        "file_name",
        F.col("file_path"),
        F.col("effective_data_date").alias("data_date"),
        F.col("file_size_bytes"),
        F.col("actual_content_hash").alias("content_hash")
    )
    .dropDuplicates(["dataset", "file_name"])
)

display(registration_candidates_df)

print(
    "New registration candidates:",
    registration_candidates_df.count()
)


dataset,file_name,file_path,data_date,file_size_bytes,content_hash


New registration candidates: 0


In [0]:

# CELL 11 — RUN-MODE SELECTION SUMMARY (READ-ONLY)

print(f"Run mode: {RUN_MODE}")
print(f"Reprocess successful: {REPROCESS_SUCCESSFUL}")
print(f"Discovered files: {hashed_files_df.count()}")
print(f"Registered files: {reconciliation_df.filter(F.col('discovery_status') == 'REGISTERED_MATCH').count()}")
print(f"Selected candidates: {selected_candidates_df.count()}")
print(f"New registrations: {registration_candidates_df.count()}")

display(
    selector_df.select(
        "file_name",
        "manifest_status",
        "discovery_status",
        "selection_reason"
    )
)


Run mode: BACKFILL
Reprocess successful: True
Discovered files: 2
Registered files: 2
Selected candidates: 1
New registrations: 0


file_name,manifest_status,discovery_status,selection_reason
products_20261007.json,SUCCESS,REGISTERED_MATCH,ELIGIBLE_REPROCESS
products_20261008.json,SUCCESS,REGISTERED_MATCH,ELIGIBLE_REPROCESS


In [0]:

# CELL 12 — RUN-MODE BEHAVIOR TEST (READ-ONLY)

def preview_run_mode(mode, reprocess_successful=False):
    eligible_reasons = [
        "REGISTER_REQUIRED",
        "ELIGIBLE",
        "CONTENT_CHANGED"
    ]

    if reprocess_successful:
        eligible_reasons.append("ELIGIBLE_REPROCESS")

    if mode == "INCREMENTAL":
        result_df = selector_df.filter(
            F.col("selection_reason").isin(
                "REGISTER_REQUIRED",
                "ELIGIBLE",
                "CONTENT_CHANGED"
            )
        )

    elif mode == "BACKFILL":
        if not BACKFILL_START_DATE or not BACKFILL_END_DATE:
            raise ValueError(
                "BACKFILL requires both start and end dates. "
                "Set the backfill date widgets before testing this mode."
            )

        start_date = F.to_date(F.lit(BACKFILL_START_DATE), "yyyy-MM-dd")
        end_date = F.to_date(F.lit(BACKFILL_END_DATE), "yyyy-MM-dd")

        result_df = selector_df.filter(
            F.col("effective_data_date").between(start_date, end_date)
            & F.col("selection_reason").isin(eligible_reasons)
        )

    elif mode == "FULL_LOAD":
        result_df = selector_df.filter(
            F.col("selection_reason").isin(eligible_reasons)
        )

    else:
        raise ValueError(f"Unsupported run mode: {mode}")

    return result_df.select(
        "file_name",
        "manifest_status",
        "selection_reason"
    )


# Test modes that don't require backfill dates
for mode in ["INCREMENTAL", "FULL_LOAD"]:
    print(f"\n--- {mode} ---")
    preview_run_mode(mode).show(truncate=False)



--- INCREMENTAL ---
+---------+---------------+----------------+
|file_name|manifest_status|selection_reason|
+---------+---------------+----------------+
+---------+---------------+----------------+


--- FULL_LOAD ---
+---------+---------------+----------------+
|file_name|manifest_status|selection_reason|
+---------+---------------+----------------+
+---------+---------------+----------------+



In [0]:

# # CELL 13 — VERIFY BACKFILL DATE FILTER (READ-ONLY)

# assert RUN_MODE == "BACKFILL", (
#     f"Expected BACKFILL, got {RUN_MODE}"
# )
# assert BACKFILL_START_DATE == "2026-10-07"
# assert BACKFILL_END_DATE == "2026-10-07"

# backfill_preview_df = preview_run_mode(
#     "BACKFILL",
#     reprocess_successful=REPROCESS_SUCCESSFUL
# )

# print("Backfill date range:", BACKFILL_START_DATE, "to", BACKFILL_END_DATE)
# print("Selected files:", backfill_preview_df.count())

# display(backfill_preview_df)


Backfill date range: 2026-10-07 to 2026-10-07
Selected files: 1


file_name,manifest_status,selection_reason
products_20261007.json,SUCCESS,ELIGIBLE_REPROCESS


In [0]:

# CELL 14 — BUILD PROCESSING PLAN (READ-ONLY)

processing_plan_df = (
    backfill_preview_df
    .join(
        reconciliation_df.select(
            "file_name",
            "file_path",
            "file_size_bytes",
            "landing_data_date",
            "actual_content_hash"
        ),
        on="file_name",
        how="inner"
    )
    .withColumn("dataset", F.lit(DATASET_NAME))
    .withColumn("run_mode", F.lit(RUN_MODE))
    .withColumn("planned_action", F.lit("REPROCESS"))
    .select(
        "dataset",
        "run_mode",
        "file_name",
        "file_path",
        "landing_data_date",
        "file_size_bytes",
        "actual_content_hash",
        "manifest_status",
        "selection_reason",
        "planned_action"
    )
)

display(processing_plan_df)

print("Files in processing plan:", processing_plan_df.count())


dataset,run_mode,file_name,file_path,landing_data_date,file_size_bytes,actual_content_hash,manifest_status,selection_reason,planned_action
products,BACKFILL,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,2026-10-07,3662500,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,SUCCESS,ELIGIBLE_REPROCESS,REPROCESS


Files in processing plan: 1


In [0]:

# CELL 15 — CREATE RUN METADATA (NO DATA WRITES)

from datetime import datetime, timezone
import uuid

RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.now(timezone.utc)

# Lease duration for a future processing claim.
LEASE_DURATION_MINUTES = 60

print("Run ID:", RUN_ID)
print("Run started at (UTC):", RUN_STARTED_AT.isoformat())
print("Run mode:", RUN_MODE)
print("Planned files:", processing_plan_df.count())
print("Lease duration (minutes):", LEASE_DURATION_MINUTES)


Run ID: 2308384d-2c39-41e2-816a-4bea198b310a
Run started at (UTC): 2026-10-11T13:32:24.312404+00:00
Run mode: BACKFILL
Planned files: 1
Lease duration (minutes): 60


In [0]:

# CELL 16 — RECHECK MANIFEST BEFORE CLAIM (READ-ONLY)

claim_precheck_df = (
    spark.read.format("delta")
    .load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & F.col("file_name").isin(
            [row["file_name"] for row in processing_plan_df.select("file_name").collect()]
        )
    )
    .select(
        "dataset",
        "file_name",
        "status",
        "content_hash",
        "run_id",
        "claimed_at",
        "lease_expires_at",
        "retry_count",
        "attempt_count"
    )
)

display(claim_precheck_df)

precheck_count = claim_precheck_df.count()
print("Matching manifest records:", precheck_count)

assert precheck_count == 1, (
    "Expected exactly one manifest record for the selected file."
)

precheck_row = claim_precheck_df.first()

assert precheck_row["status"] == "SUCCESS", (
    f"Status changed to {precheck_row['status']}; rebuild the plan before proceeding."
)

assert precheck_row["content_hash"] == processing_plan_df.first()["actual_content_hash"], (
    "Manifest hash differs from the discovered file; stop and investigate."
)

print("PASS: Manifest state and content hash match the plan.")


dataset,file_name,status,content_hash,run_id,claimed_at,lease_expires_at,retry_count,attempt_count
products,products_20261007.json,SUCCESS,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T11:37:52.030Z,2026-10-11T12:37:52.030Z,0,2


Matching manifest records: 1
PASS: Manifest state and content hash match the plan.


In [0]:

# CELL 17 — VERIFY EXISTING BRONZE DATA (READ-ONLY)

bronze_df = spark.read.format("delta").load(BRONZE_PATH)

target_file = "products_20261007.json"

existing_file_df = bronze_df.filter(
    F.col("_source_file") == target_file
)

print("Target file:", target_file)
print("Existing Bronze rows:", existing_file_df.count())

display(
    existing_file_df.select(
        "_source_file",
        "_dataset",
        "_batch_id",
        "_run_id",
        "_ingested_at"
    ).limit(5)
)


Target file: products_20261007.json
Existing Bronze rows: 25000


_source_file,_dataset,_batch_id,_run_id,_ingested_at
products_20261007.json,products,9df746d8-611b-493a-adeb-94f07054166a,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:30:55.977Z
products_20261007.json,products,9df746d8-611b-493a-adeb-94f07054166a,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:30:55.977Z
products_20261007.json,products,9df746d8-611b-493a-adeb-94f07054166a,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:30:55.977Z
products_20261007.json,products,9df746d8-611b-493a-adeb-94f07054166a,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:30:55.977Z
products_20261007.json,products,9df746d8-611b-493a-adeb-94f07054166a,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:30:55.977Z


In [0]:

# CELL 18 — BRONZE INVENTORY (READ-ONLY)

bronze_inventory_df = (
    bronze_df
    .groupBy("_source_file", "_dataset")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("product_id").alias("distinct_product_ids"),
        F.sum(
            F.when(F.col("product_id").isNull(), 1).otherwise(0)
        ).alias("null_product_ids")
    )
    .orderBy("_source_file")
)

display(bronze_inventory_df)

target_inventory_df = bronze_inventory_df.filter(
    F.col("_source_file") == target_file
)

target_inventory = target_inventory_df.first()

assert target_inventory is not None, "Target file is missing from Bronze."
assert target_inventory["record_count"] == 25000, (
    f"Expected 25000 records, found {target_inventory['record_count']}."
)
assert target_inventory["null_product_ids"] == 0, (
    "Null product IDs found; stop before reprocessing."
)

print("PASS: Existing Bronze inventory verified.")


_source_file,_dataset,record_count,distinct_product_ids,null_product_ids
products_20261007.json,products,25000,25000,0
products_20261008.json,products,25000,25000,0


PASS: Existing Bronze inventory verified.


In [0]:

# CELL 19 — REPLACEMENT SCOPE CHECK (READ-ONLY)

target_rows = bronze_df.filter(
    F.col("_source_file") == target_file
).count()

other_rows = bronze_df.filter(
    F.col("_source_file") != target_file
).count()

unknown_source_rows = bronze_df.filter(
    F.col("_source_file").isNull()
).count()

print("Target file rows:", target_rows)
print("Other source rows:", other_rows)
print("Rows with null source file:", unknown_source_rows)

assert target_rows == 25000
assert other_rows == 25000
assert unknown_source_rows == 0

print("PASS: Replacement scope is isolated to the target file.")


Target file rows: 25000
Other source rows: 25000
Rows with null source file: 0
PASS: Replacement scope is isolated to the target file.


In [0]:

# CELL 20 — SOURCE VALIDATION BEFORE BACKFILL (READ-ONLY)

source_file_path = processing_plan_df.select("file_path").first()["file_path"]

source_df = (
    spark.read
    .schema(PRODUCTS_SCHEMA)
    .option("multiLine", "true")
    .option("mode", "FAILFAST")
    .json(source_file_path)
)

source_count = source_df.count()
source_distinct_ids = source_df.select("product_id").distinct().count()

source_null_ids = source_df.filter(
    F.col("product_id").isNull()
).count()

source_negative_prices = source_df.filter(
    F.col("price") < 0
).count()

print("Source file:", target_file)
print("Source records:", source_count)
print("Distinct product IDs:", source_distinct_ids)
print("Null product IDs:", source_null_ids)
print("Negative prices:", source_negative_prices)

assert source_count == 25000
assert source_distinct_ids == source_count
assert source_null_ids == 0
assert source_negative_prices == 0

print("PASS: Source validation completed.")


Source file: products_20261007.json
Source records: 25000
Distinct product IDs: 25000
Null product IDs: 0
Negative prices: 0
PASS: Source validation completed.


In [0]:

# CELL 21 — DEFINE BACKFILL WRITE PLAN (NO WRITES)

BACKFILL_WRITE_STRATEGY = "REPLACE_BY_SOURCE_FILE"

backfill_write_plan = {
    "run_id": RUN_ID,
    "dataset": DATASET_NAME,
    "source_file": target_file,
    "expected_source_rows": source_count,
    "expected_existing_rows": target_rows,
    "expected_other_rows": other_rows,
    "strategy": BACKFILL_WRITE_STRATEGY,
    "manifest_write_enabled": False,
    "bronze_write_enabled": False
}

for key, value in backfill_write_plan.items():
    print(f"{key}: {value}")

assert BACKFILL_WRITE_STRATEGY == "REPLACE_BY_SOURCE_FILE"
assert target_rows == source_count
assert other_rows == 25000

print("PASS: Backfill plan prepared; no writes performed.")


run_id: 2308384d-2c39-41e2-816a-4bea198b310a
dataset: products
source_file: products_20261007.json
expected_source_rows: 25000
expected_existing_rows: 25000
expected_other_rows: 25000
strategy: REPLACE_BY_SOURCE_FILE
manifest_write_enabled: False
bronze_write_enabled: False
PASS: Backfill plan prepared; no writes performed.


In [0]:

# CELL 22 — MANUAL EXECUTION SAFETY GATE (READ-ONLY)

assert RUN_MODE == "BACKFILL", (
    f"Expected BACKFILL mode, got {RUN_MODE}"
)

assert REPROCESS_SUCCESSFUL, (
    "Backfill reprocessing must be explicitly enabled."
)

assert processing_plan_df.count() == 1, (
    "Expected exactly one file in the processing plan."
)

assert target_file == "products_20261007.json"

assert source_count == 25000
assert target_rows == 25000
assert other_rows == 25000

print("PASS: Manual execution checks passed.")
print("Dataset:", DATASET_NAME)
print("Target file:", target_file)
print("Run ID:", RUN_ID)
print("WARNING: Do not run another writer against this dataset concurrently.")
print("No manifest or Bronze writes performed.")


PASS: Manual execution checks passed.
Dataset: products
Target file: products_20261007.json
Run ID: 2308384d-2c39-41e2-816a-4bea198b310a
No manifest or Bronze writes performed.


In [0]:

# CELL 23 — BRONZE TABLE METADATA (READ-ONLY)

from delta.tables import DeltaTable

bronze_delta = DeltaTable.forPath(spark, BRONZE_PATH)

history_df = (
    bronze_delta.history(5)
    .select(
        "version",
        "timestamp",
        "operation",
        "operationParameters"
    )
    .orderBy(F.col("version").desc())
)

display(history_df)

current_version = (
    bronze_delta.history(1)
    .select("version")
    .first()["version"]
)

print("Bronze path:", BRONZE_PATH)
print("Current Delta version:", current_version)
print("Target file:", target_file)
print("Expected target rows:", target_rows)
print("Expected other rows:", other_rows)


version,timestamp,operation,operationParameters
1,2026-10-11T12:30:59.000Z,MERGE,"Map(predicate -> [""((_source_file#41667 = _source_file#41675) AND (product_id#41662L = product_id#41560L))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])"
0,2026-10-11T12:05:59.000Z,WRITE,"Map(mode -> ErrorIfExists, statsOnLoad -> false, partitionBy -> [])"


Bronze path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/
Current Delta version: 1
Target file: products_20261007.json
Expected target rows: 25000
Expected other rows: 25000


In [0]:

# CELL 24 — PREPARE MANIFEST CLAIM REQUEST (NO WRITES)

CLAIMED_AT = datetime.now(timezone.utc)
LEASE_EXPIRES_AT = CLAIMED_AT + __import__("datetime").timedelta(
    minutes=LEASE_DURATION_MINUTES
)

claim_request = {
    "dataset": DATASET_NAME,
    "file_name": target_file,
    "expected_status": "SUCCESS",
    "expected_content_hash": processing_plan_df.first()[
        "actual_content_hash"
    ],
    "new_run_id": RUN_ID,
    "claimed_at": CLAIMED_AT.isoformat(),
    "lease_expires_at": LEASE_EXPIRES_AT.isoformat(),
    "expected_bronze_version": current_version
}

for key, value in claim_request.items():
    print(f"{key}: {value}")

assert claim_request["expected_status"] == "SUCCESS"
assert len(claim_request["expected_content_hash"]) == 64
assert claim_request["expected_bronze_version"] == 1

print("PASS: Claim request prepared; manifest unchanged.")


dataset: products
file_name: products_20261007.json
expected_status: SUCCESS
expected_content_hash: 70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118
new_run_id: 2308384d-2c39-41e2-816a-4bea198b310a
claimed_at: 2026-10-11T13:40:24.160813+00:00
lease_expires_at: 2026-10-11T14:40:24.160813+00:00
expected_bronze_version: 1
PASS: Claim request prepared; manifest unchanged.


In [0]:

# CELL 25 — FINAL MANIFEST PRECHECK (READ-ONLY)

latest_manifest_row = (
    spark.read.format("delta")
    .load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("file_name") == target_file)
    )
    .select(
        "status",
        "content_hash",
        "run_id",
        "retry_count",
        "attempt_count"
    )
    .first()
)

assert latest_manifest_row is not None, "Manifest record not found."

assert latest_manifest_row["status"] == "SUCCESS", (
    f"Expected SUCCESS; found {latest_manifest_row['status']}"
)

assert latest_manifest_row["content_hash"] == claim_request[
    "expected_content_hash"
], "Manifest content hash changed."

assert latest_manifest_row["run_id"] != RUN_ID, (
    "This run ID is already assigned; inspect the manifest before proceeding."
)

print("Current status:", latest_manifest_row["status"])
print("Current run ID:", latest_manifest_row["run_id"])
print("Retry count:", latest_manifest_row["retry_count"])
print("Attempt count:", latest_manifest_row["attempt_count"])
print("PASS: Final manifest precheck.")
print("No writes performed.")


Current status: SUCCESS
Current run ID: eebf67b1-6ce3-4250-bc99-e6fca60f341d
Retry count: 0
Attempt count: 2
PASS: Final manifest precheck.
No writes performed.


In [0]:

# CELL 26 — INSPECT TARGET MANIFEST SCHEMA (READ-ONLY)

manifest_schema_df = spark.read.format("delta").load(MANIFEST_PATH)

print("Manifest schema:")
manifest_schema_df.printSchema()

print("\nTarget record:")
display(
    manifest_schema_df.filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("file_name") == target_file)
    )
)


Manifest schema:
root
 |-- dataset: string (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_path: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- first_seen: timestamp (nullable = true)
 |-- processed_at: timestamp (nullable = true)
 |-- run_id: string (nullable = true)
 |-- retry_count: integer (nullable = true)
 |-- error_message: string (nullable = true)
 |-- data_date: date (nullable = true)
 |-- file_size_bytes: long (nullable = true)
 |-- source_modified_at: timestamp (nullable = true)
 |-- content_hash: string (nullable = true)
 |-- last_seen_at: timestamp (nullable = true)
 |-- discovery_run_id: string (nullable = true)
 |-- claimed_at: timestamp (nullable = true)
 |-- lease_expires_at: timestamp (nullable = true)
 |-- last_attempt_at: timestamp (nullable = true)
 |-- attempt_count: integer (nullable = true)
 |-- last_success_at: timestamp (nullable = true)
 |-- status_updated_at: timestamp (nullabl

dataset,file_name,file_path,batch_id,status,first_seen,processed_at,run_id,retry_count,error_message,data_date,file_size_bytes,source_modified_at,content_hash,last_seen_at,discovery_run_id,claimed_at,lease_expires_at,last_attempt_at,attempt_count,last_success_at,status_updated_at,source_record_count,written_record_count,rejected_record_count,processed_content_hash,schema_fingerprint,target_path,target_delta_version,processing_duration_seconds,error_type,last_error_at
products,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,9df746d8-611b-493a-adeb-94f07054166a,SUCCESS,2026-10-11T09:38:16.334Z,2026-10-11T12:11:11.095Z,eebf67b1-6ce3-4250-bc99-e6fca60f341d,0,null,2026-10-07,3662500,2026-10-07T08:21:24.000Z,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,2026-10-11T09:29:27.360Z,DISCOVERY_20261011092927,2026-10-11T11:37:52.030Z,2026-10-11T12:37:52.030Z,2026-10-11T11:37:52.030Z,2,null,2026-10-11T12:11:11.095Z,25000,25000,null,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,null,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/,0,null,null,null


In [0]:

# CELL 27 — PREPARE CLAIM VALUES (NO WRITES)

claim_values = {
    "dataset": DATASET_NAME,
    "file_name": target_file,
    "expected_status": "SUCCESS",
    "expected_content_hash": claim_request["expected_content_hash"],
    "new_run_id": RUN_ID,
    "claimed_at": CLAIMED_AT,
    "lease_expires_at": LEASE_EXPIRES_AT,
    "new_attempt_count": (latest_manifest_row["attempt_count"] or 0) + 1,
    "new_retry_count": latest_manifest_row["retry_count"] or 0
}

for key, value in claim_values.items():
    print(f"{key}: {value}")

assert claim_values["new_run_id"] != latest_manifest_row["run_id"]
assert claim_values["expected_status"] == "SUCCESS"
assert claim_values["new_attempt_count"] == 3

print("PASS: Claim values prepared; real manifest unchanged.")


dataset: products
file_name: products_20261007.json
expected_status: SUCCESS
expected_content_hash: 70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118
new_run_id: 2308384d-2c39-41e2-816a-4bea198b310a
claimed_at: 2026-10-11 13:40:24.160813+00:00
lease_expires_at: 2026-10-11 14:40:24.160813+00:00
new_attempt_count: 3
new_retry_count: 0
PASS: Claim values prepared; real manifest unchanged.


In [0]:

# CELL 28 — FINAL WRITE-READINESS CHECK (READ-ONLY)

latest_row = (
    spark.read.format("delta")
    .load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("file_name") == target_file)
    )
    .first()
)

assert latest_row is not None, "Manifest record is missing."
assert latest_row["status"] == "SUCCESS", "Manifest status changed."
assert latest_row["content_hash"] == claim_values["expected_content_hash"], (
    "Manifest hash changed."
)
assert latest_row["run_id"] == latest_manifest_row["run_id"], (
    "Manifest run_id changed since precheck."
)
assert source_count == 25000
assert target_rows == 25000
assert other_rows == 25000

print("PASS: Manifest and source preconditions still hold.")
print("Target file:", target_file)
print("Source records:", source_count)
print("Existing target rows:", target_rows)
print("Other file rows:", other_rows)
print("No writes performed.")


PASS: Manifest and source preconditions still hold.
Target file: products_20261007.json
Source records: 25000
Existing target rows: 25000
Other file rows: 25000
No writes performed.


In [0]:

# CELL 29 — CLAIM & RECOVERY PROTOCOL (NO WRITES)

CLAIM_PROTOCOL = {
    "allowed_initial_status": "SUCCESS",
    "claim_status": "PROCESSING",
    "claim_requires_matching_content_hash": True,
    "claim_records_run_id": True,
    "claim_records_lease": True,
    "claim_increments_attempt_count": True,
    "claim_increments_retry_count": False,
    "bronze_strategy": "REPLACE_BY_SOURCE_FILE",
    "preserve_other_source_files": True,
    "final_status_after_verified_write": "SUCCESS",
    "recover_after_bronze_write": "VERIFY_BRONZE_THEN_FINALIZE_MANIFEST",
    "concurrent_writer_policy": "SINGLE_WRITER_REQUIRED",
    "manifest_bronze_atomic_transaction": False
}

for key, value in CLAIM_PROTOCOL.items():
    print(f"{key}: {value}")

assert CLAIM_PROTOCOL["allowed_initial_status"] == "SUCCESS"
assert CLAIM_PROTOCOL["claim_status"] == "PROCESSING"
assert CLAIM_PROTOCOL["bronze_strategy"] == "REPLACE_BY_SOURCE_FILE"
assert CLAIM_PROTOCOL["preserve_other_source_files"] is True
assert CLAIM_PROTOCOL["manifest_bronze_atomic_transaction"] is False

print("PASS: Claim and recovery protocol defined.")
print("No manifest or Bronze writes performed.")



allowed_initial_status: SUCCESS
claim_status: PROCESSING
claim_requires_matching_content_hash: True
claim_records_run_id: True
claim_records_lease: True
claim_increments_attempt_count: True
claim_increments_retry_count: False
bronze_strategy: REPLACE_BY_SOURCE_FILE
preserve_other_source_files: True
final_status_after_verified_write: SUCCESS
recover_after_bronze_write: VERIFY_BRONZE_THEN_FINALIZE_MANIFEST
concurrent_writer_policy: SINGLE_WRITER_REQUIRED
manifest_bronze_atomic_transaction: False
PASS: Claim and recovery protocol defined.
No manifest or Bronze writes performed.


In [0]:

# CELL 30 — MANUAL EXECUTION GUARD (NO WRITES)

if "BACKFILL_EXECUTION_GUARD" not in globals():
    BACKFILL_EXECUTION_GUARD = {
        "run_id": RUN_ID,
        "dataset": DATASET_NAME,
        "file_name": target_file,
        "claim_started": False,
        "bronze_write_started": False,
        "finalized": False
    }

assert BACKFILL_EXECUTION_GUARD["run_id"] == RUN_ID
assert BACKFILL_EXECUTION_GUARD["dataset"] == DATASET_NAME
assert BACKFILL_EXECUTION_GUARD["file_name"] == target_file

assert BACKFILL_EXECUTION_GUARD["claim_started"] is False, (
    "A claim has already started in this notebook session. "
    "Inspect the manifest before retrying."
)

print("Execution guard initialized.")
print("Run ID:", BACKFILL_EXECUTION_GUARD["run_id"])
print("Target:", BACKFILL_EXECUTION_GUARD["file_name"])
print("Claim started:", BACKFILL_EXECUTION_GUARD["claim_started"])
print("Bronze write started:", BACKFILL_EXECUTION_GUARD["bronze_write_started"])
print("Finalized:", BACKFILL_EXECUTION_GUARD["finalized"])
print("PASS: No writes performed.")


Execution guard initialized.
Run ID: 2308384d-2c39-41e2-816a-4bea198b310a
Target: products_20261007.json
Claim started: False
Bronze write started: False
Finalized: False
PASS: No writes performed.


In [0]:

# CELL 31 — CLAIM PRECONDITIONS REVIEW (NO WRITES)

claim_check_df = (
    spark.read.format("delta")
    .load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("file_name") == target_file)
    )
    .select(
        "dataset",
        "file_name",
        "status",
        "content_hash",
        "run_id",
        "retry_count",
        "attempt_count"
    )
)

claim_check_rows = claim_check_df.collect()

assert len(claim_check_rows) == 1, (
    f"Expected one manifest row, found {len(claim_check_rows)}."
)

claim_check = claim_check_rows[0]

assert claim_check["status"] == "SUCCESS", (
    f"Cannot claim file in status {claim_check['status']}."
)

assert claim_check["content_hash"] == claim_values["expected_content_hash"], (
    "Manifest hash does not match the selected source."
)

assert claim_check["run_id"] != RUN_ID, (
    "The current run ID is already assigned to this record."
)

assert not BACKFILL_EXECUTION_GUARD["claim_started"], (
    "This notebook session has already started a claim."
)

print("Claim preconditions: PASS")
print("Current status:", claim_check["status"])
print("Current run ID:", claim_check["run_id"])
print("Current attempt count:", claim_check["attempt_count"])
print("Current retry count:", claim_check["retry_count"])
print("No writes performed.")


Claim preconditions: PASS
Current status: SUCCESS
Current run ID: eebf67b1-6ce3-4250-bc99-e6fca60f341d
Current attempt count: 2
Current retry count: 0
No writes performed.


In [0]:

# CELL 32 — CLAIM PREVIEW (NO WRITES)

from datetime import datetime, timezone, timedelta

claim_now = datetime.now(timezone.utc)
claim_lease_expiry = claim_now + timedelta(
    minutes=LEASE_DURATION_MINUTES
)

claim_preview = {
    "dataset": DATASET_NAME,
    "file_name": target_file,
    "expected_status": "SUCCESS",
    "expected_content_hash": claim_values["expected_content_hash"],
    "new_status": "PROCESSING",
    "new_run_id": RUN_ID,
    "claimed_at": claim_now.isoformat(),
    "lease_expires_at": claim_lease_expiry.isoformat(),
    "new_attempt_count": (claim_check["attempt_count"] or 0) + 1,
    "new_retry_count": claim_check["retry_count"] or 0
}

for key, value in claim_preview.items():
    print(f"{key}: {value}")

assert claim_preview["new_status"] == "PROCESSING"
assert claim_preview["new_attempt_count"] == 3
assert claim_preview["new_retry_count"] == 0
assert claim_preview["expected_content_hash"] == (
    claim_check["content_hash"]
)

print("PASS: Claim preview is internally consistent.")
print("Manifest unchanged; no writes performed.")


dataset: products
file_name: products_20261007.json
expected_status: SUCCESS
expected_content_hash: 70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118
new_status: PROCESSING
new_run_id: 2308384d-2c39-41e2-816a-4bea198b310a
claimed_at: 2026-10-11T13:47:38.444152+00:00
lease_expires_at: 2026-10-11T14:47:38.444152+00:00
new_attempt_count: 3
new_retry_count: 0
PASS: Claim preview is internally consistent.
Manifest unchanged; no writes performed.


In [0]:

# CELL 33 — CLAIM TARGET UNIQUENESS CHECK (READ-ONLY)

matching_target_count = (
    spark.read.format("delta")
    .load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == claim_preview["dataset"])
        & (F.col("file_name") == claim_preview["file_name"])
        & (F.col("status") == claim_preview["expected_status"])
        & (
            F.col("content_hash")
            == claim_preview["expected_content_hash"]
        )
    )
    .count()
)

print("Matching claim targets:", matching_target_count)

assert matching_target_count == 1, (
    "Claim must stop unless exactly one matching manifest record exists."
)

print("PASS: Exactly one record matches the proposed claim.")
print("No writes performed.")


Matching claim targets: 1
PASS: Exactly one record matches the proposed claim.
No writes performed.


In [0]:

# CELL 34 — CLAIM TARGET FILE IN MANIFEST (WRITE)
# Assumption: single writer, as confirmed by the user.

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from datetime import datetime, timezone, timedelta

assert not BACKFILL_EXECUTION_GUARD["claim_started"], (
    "Claim already started in this notebook session. Inspect manifest first."
)

# Fresh timestamps at execution time.
claim_now = datetime.now(timezone.utc)
claim_expiry = claim_now + timedelta(minutes=LEASE_DURATION_MINUTES)

manifest_delta = DeltaTable.forPath(spark, MANIFEST_PATH)

# Conditional update of exactly the intended record.
(
    manifest_delta.alias("t")
    .update(
        condition=(
            (F.col("t.dataset") == DATASET_NAME)
            & (F.col("t.file_name") == target_file)
            & (F.col("t.status") == "SUCCESS")
            & (
                F.col("t.content_hash")
                == claim_values["expected_content_hash"]
            )
        ),
        set={
            "status": F.lit("PROCESSING"),
            "run_id": F.lit(RUN_ID),
            "claimed_at": F.lit(claim_now).cast("timestamp"),
            "lease_expires_at": F.lit(claim_expiry).cast("timestamp"),
            "last_attempt_at": F.lit(claim_now).cast("timestamp"),
            "attempt_count": F.coalesce(
                F.col("t.attempt_count"), F.lit(0)
            ) + F.lit(1),
            "status_updated_at": F.lit(claim_now).cast("timestamp"),
            "error_message": F.lit(None).cast("string"),
            "error_type": F.lit(None).cast("string")
        }
    )
)

# Read back and verify ownership.
claimed_row = (
    spark.read.format("delta")
    .load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("file_name") == target_file)
    )
    .select(
        "status", "run_id", "content_hash",
        "claimed_at", "lease_expires_at",
        "retry_count", "attempt_count"
    )
    .first()
)

assert claimed_row is not None, "Claimed manifest row not found."
assert claimed_row["status"] == "PROCESSING", (
    f"Expected PROCESSING, found {claimed_row['status']}"
)
assert claimed_row["run_id"] == RUN_ID, (
    "Claim ownership verification failed. STOP."
)
assert claimed_row["content_hash"] == claim_values["expected_content_hash"], (
    "Manifest hash changed. STOP."
)

BACKFILL_EXECUTION_GUARD["claim_started"] = True

print("PASS: Manifest claim verified.")
print("File:", target_file)
print("Status:", claimed_row["status"])
print("Run ID:", claimed_row["run_id"])
print("Attempt count:", claimed_row["attempt_count"])
print("Retry count:", claimed_row["retry_count"])
print("Lease expires:", claimed_row["lease_expires_at"])
print("Bronze data has not been modified.")


PASS: Manifest claim verified.
File: products_20261007.json
Status: PROCESSING
Run ID: 2308384d-2c39-41e2-816a-4bea198b310a
Attempt count: 3
Retry count: 0
Lease expires: 2026-10-11 14:49:28.442330
Bronze data has not been modified.


In [0]:

# CELL 35 — VERIFY CLAIM OWNERSHIP AND SOURCE HASH (READ-ONLY)

# Re-read manifest ownership.
ownership_row = (
    spark.read.format("delta")
    .load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("file_name") == target_file)
    )
    .select("status", "run_id", "content_hash", "lease_expires_at")
    .first()
)

assert ownership_row is not None, "Manifest row missing."
assert ownership_row["status"] == "PROCESSING", "File is not PROCESSING."
assert ownership_row["run_id"] == RUN_ID, "Ownership mismatch. STOP."
assert ownership_row["content_hash"] == claim_values[
    "expected_content_hash"
], "Manifest content hash mismatch. STOP."

# Recompute the source hash from storage.
current_source_hash = (
    spark.read.format("binaryFile")
    .load(source_file_path)
    .select(F.sha2(F.col("content"), 256).alias("hash"))
    .first()["hash"]
)

assert current_source_hash == claim_values["expected_content_hash"], (
    "Source content changed after planning. STOP."
)

print("PASS: Claim ownership verified.")
print("PASS: Source content hash verified.")
print("File:", target_file)
print("Run ID:", RUN_ID)
print("No Bronze writes performed.")


PASS: Claim ownership verified.
PASS: Source content hash verified.
File: products_20261007.json
Run ID: 2308384d-2c39-41e2-816a-4bea198b310a
No Bronze writes performed.


In [0]:

# CELL 36 — PREPARE TARGET-FILE BRONZE DATA (NO WRITES)

from pyspark.sql import functions as F

BATCH_ID = str(uuid.uuid4())
INGESTED_AT = datetime.now(timezone.utc).replace(tzinfo=None)

replacement_df = (
    source_df
    .withColumn("_source_file", F.lit(target_file))
    .withColumn("_source_path", F.lit(source_file_path))
    .withColumn("_dataset", F.lit(DATASET_NAME))
    .withColumn("_batch_id", F.lit(BATCH_ID))
    .withColumn("_run_id", F.lit(RUN_ID))
    .withColumn("_ingested_at", F.lit(INGESTED_AT).cast("timestamp"))
    .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
)

replacement_count = replacement_df.count()
replacement_distinct_ids = replacement_df.select("product_id").distinct().count()
replacement_null_ids = replacement_df.filter(
    F.col("product_id").isNull()
).count()

print("Target file:", target_file)
print("Batch ID:", BATCH_ID)
print("Run ID:", RUN_ID)
print("Replacement rows:", replacement_count)
print("Distinct product IDs:", replacement_distinct_ids)
print("Null product IDs:", replacement_null_ids)

assert replacement_count == 25000
assert replacement_distinct_ids == replacement_count
assert replacement_null_ids == 0

print("PASS: Replacement DataFrame validated.")
print("No Bronze writes performed.")


Target file: products_20261007.json
Batch ID: 4ebccb45-0586-4fa4-ae40-72d4a5b7c581
Run ID: 2308384d-2c39-41e2-816a-4bea198b310a
Replacement rows: 25000
Distinct product IDs: 25000
Null product IDs: 0
PASS: Replacement DataFrame validated.
No Bronze writes performed.


In [0]:
(
    replacement_df.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"_source_file = '{target_file}'")
    .save(BRONZE_PATH)
)

In [0]:

# CELL 37 — REPLACE TARGET FILE IN BRONZE (WRITE)
# Assumption: no concurrent writers, as confirmed.

# 1. Recheck manifest ownership immediately before writing.
write_ownership = (
    spark.read.format("delta")
    .load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("file_name") == target_file)
    )
    .select("status", "run_id", "content_hash")
    .first()
)

assert write_ownership is not None, "Manifest record missing."
assert write_ownership["status"] == "PROCESSING", "Claim status changed."
assert write_ownership["run_id"] == RUN_ID, "Claim ownership lost."
assert write_ownership["content_hash"] == current_source_hash, (
    "Source hash no longer matches the manifest."
)

# 2. Confirm current target and non-target counts.
pre_write_bronze = spark.read.format("delta").load(BRONZE_PATH)

pre_target_count = pre_write_bronze.filter(
    F.col("_source_file") == target_file
).count()

pre_other_count = pre_write_bronze.filter(
    F.col("_source_file") != target_file
).count()

assert pre_target_count == 25000, (
    f"Unexpected target count: {pre_target_count}"
)
assert pre_other_count == 25000, (
    f"Unexpected non-target count: {pre_other_count}"
)

# 3. Replace only the target file's rows.
(
    replacement_df.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"_source_file = '{target_file}'")
    .save(BRONZE_PATH)
)

# 4. Capture the resulting Delta version.
post_write_version = (
    DeltaTable.forPath(spark, BRONZE_PATH)
    .history(1)
    .select("version", "operation")
    .first()
)

BACKFILL_EXECUTION_GUARD["bronze_write_started"] = True

print("Bronze replacement command completed.")
print("Target file:", target_file)
print("Run ID:", RUN_ID)
print("Previous target rows:", pre_target_count)
print("Previous other rows:", pre_other_count)
print("Latest Delta version:", post_write_version["version"])
print("Latest operation:", post_write_version["operation"])
print("Next: run the separate verification cell.")



Bronze replacement command completed.
Target file: products_20261007.json
Run ID: 2308384d-2c39-41e2-816a-4bea198b310a
Previous target rows: 25000
Previous other rows: 25000
Latest Delta version: 3
Latest operation: WRITE
Next: run the separate verification cell.


In [0]:

# CELL 38 — POST-WRITE BRONZE VERIFICATION (READ-ONLY)

post_bronze_df = spark.read.format("delta").load(BRONZE_PATH)

# Count records by source file.
post_target_df = post_bronze_df.filter(
    F.col("_source_file") == target_file
)

post_other_df = post_bronze_df.filter(
    F.col("_source_file") == "products_20261008.json"
)

post_target_count = post_target_df.count()
post_target_distinct_ids = post_target_df.select("product_id").distinct().count()
post_target_null_ids = post_target_df.filter(
    F.col("product_id").isNull()
).count()

post_other_count = post_other_df.count()
post_total_count = post_bronze_df.count()

wrong_run_count = post_target_df.filter(
    F.col("_run_id") != RUN_ID
).count()

wrong_batch_count = post_target_df.filter(
    F.col("_batch_id") != BATCH_ID
).count()

print("October 7 rows:", post_target_count)
print("October 7 distinct product IDs:", post_target_distinct_ids)
print("October 7 null product IDs:", post_target_null_ids)
print("October 7 rows with wrong run ID:", wrong_run_count)
print("October 7 rows with wrong batch ID:", wrong_batch_count)
print("October 8 rows:", post_other_count)
print("Total Bronze rows:", post_total_count)

assert post_target_count == 25000
assert post_target_distinct_ids == 25000
assert post_target_null_ids == 0
assert wrong_run_count == 0
assert wrong_batch_count == 0
assert post_other_count == 25000
assert post_total_count == 50000

print("PASS: Bronze replacement verified.")
print("Manifest finalization has not been performed.")


October 7 rows: 25000
October 7 distinct product IDs: 25000
October 7 null product IDs: 0
October 7 rows with wrong run ID: 0
October 7 rows with wrong batch ID: 0
October 8 rows: 25000
Total Bronze rows: 50000
PASS: Bronze replacement verified.
Manifest finalization has not been performed.


In [0]:

# CELL 39 — PREPARE FINALIZATION (NO WRITES)

from datetime import datetime, timezone

FINALIZED_AT = datetime.now(timezone.utc)

finalization_values = {
    "dataset": DATASET_NAME,
    "file_name": target_file,
    "expected_status": "PROCESSING",
    "expected_run_id": RUN_ID,
    "final_status": "SUCCESS",
    "processed_content_hash": current_source_hash,
    "source_record_count": source_count,
    "written_record_count": post_target_count,
    "rejected_record_count": 0,
    "target_path": BRONZE_PATH,
    "target_delta_version": post_write_version["version"],
    "batch_id": BATCH_ID,
    "processed_at": FINALIZED_AT.isoformat()
}

for key, value in finalization_values.items():
    print(f"{key}: {value}")

assert finalization_values["final_status"] == "SUCCESS"
assert finalization_values["source_record_count"] == 25000
assert finalization_values["written_record_count"] == 25000
assert finalization_values["target_delta_version"] == 3

print("PASS: Finalization values prepared.")
print("Manifest has not been updated.")


dataset: products
file_name: products_20261007.json
expected_status: PROCESSING
expected_run_id: 2308384d-2c39-41e2-816a-4bea198b310a
final_status: SUCCESS
processed_content_hash: 70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118
source_record_count: 25000
written_record_count: 25000
rejected_record_count: 0
target_path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/
target_delta_version: 3
batch_id: 4ebccb45-0586-4fa4-ae40-72d4a5b7c581
processed_at: 2026-10-11T13:53:25.942338+00:00
PASS: Finalization values prepared.
Manifest has not been updated.


In [0]:

# CELL 40 — FINALIZE MANIFEST (WRITE)

# 1. Recheck ownership immediately before finalization.
manifest_delta = DeltaTable.forPath(spark, MANIFEST_PATH)

current_owner = (
    spark.read.format("delta")
    .load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("file_name") == target_file)
    )
    .select("status", "run_id", "content_hash")
    .first()
)

assert current_owner is not None, "Manifest row missing."
assert current_owner["status"] == "PROCESSING", (
    f"Expected PROCESSING, found {current_owner['status']}"
)
assert current_owner["run_id"] == RUN_ID, (
    "This run no longer owns the manifest record."
)
assert current_owner["content_hash"] == current_source_hash, (
    "Manifest content hash changed."
)

# 2. Update only the target row owned by this run.
finalized_at = datetime.now(timezone.utc)

(
    manifest_delta.alias("t")
    .update(
        condition=(
            (F.col("t.dataset") == DATASET_NAME)
            & (F.col("t.file_name") == target_file)
            & (F.col("t.status") == "PROCESSING")
            & (F.col("t.run_id") == RUN_ID)
            & (
                F.col("t.content_hash")
                == current_source_hash
            )
        ),
        set={
            "status": F.lit("SUCCESS"),
            "run_id": F.lit(RUN_ID),
            "batch_id": F.lit(BATCH_ID),
            "processed_at": F.lit(finalized_at).cast("timestamp"),
            "last_success_at": F.lit(finalized_at).cast("timestamp"),
            "status_updated_at": F.lit(finalized_at).cast("timestamp"),
            "processed_content_hash": F.lit(current_source_hash),
            "source_record_count": F.lit(source_count).cast("long"),
            "written_record_count": F.lit(post_target_count).cast("long"),
            "rejected_record_count": F.lit(0).cast("long"),
            "target_path": F.lit(BRONZE_PATH),
            "target_delta_version": F.lit(
                post_write_version["version"]
            ).cast("long"),
            "error_message": F.lit(None).cast("string"),
            "error_type": F.lit(None).cast("string"),
            "last_error_at": F.lit(None).cast("timestamp"),
            "lease_expires_at": F.lit(None).cast("timestamp")
        }
    )
)

print("Manifest finalization update executed.")
print("Next: run the separate verification cell.")


Manifest finalization update executed.
Next: run the separate verification cell.


In [0]:

# CELL 41 — VERIFY FINAL MANIFEST STATE (READ-ONLY)

final_manifest_row = (
    spark.read.format("delta")
    .load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("file_name") == target_file)
    )
    .select(
        "dataset",
        "file_name",
        "status",
        "run_id",
        "batch_id",
        "content_hash",
        "processed_content_hash",
        "source_record_count",
        "written_record_count",
        "rejected_record_count",
        "target_path",
        "target_delta_version",
        "attempt_count",
        "retry_count",
        "lease_expires_at",
        "processed_at"
    )
    .first()
)

assert final_manifest_row is not None, "Manifest row not found."
assert final_manifest_row["status"] == "SUCCESS", (
    f"Expected SUCCESS, found {final_manifest_row['status']}"
)
assert final_manifest_row["run_id"] == RUN_ID, "Run ID mismatch."
assert final_manifest_row["batch_id"] == BATCH_ID, "Batch ID mismatch."
assert final_manifest_row["content_hash"] == current_source_hash
assert final_manifest_row["processed_content_hash"] == current_source_hash
assert final_manifest_row["source_record_count"] == 25000
assert final_manifest_row["written_record_count"] == 25000
assert final_manifest_row["rejected_record_count"] == 0
assert final_manifest_row["target_path"] == BRONZE_PATH
assert final_manifest_row["target_delta_version"] == 3
assert final_manifest_row["lease_expires_at"] is None

print("Final manifest status:", final_manifest_row["status"])
print("Run ID:", final_manifest_row["run_id"])
print("Batch ID:", final_manifest_row["batch_id"])
print("Source records:", final_manifest_row["source_record_count"])
print("Written records:", final_manifest_row["written_record_count"])
print("Rejected records:", final_manifest_row["rejected_record_count"])
print("Target Delta version:", final_manifest_row["target_delta_version"])
print("Lease cleared:", final_manifest_row["lease_expires_at"] is None)

print("PASS: Manifest finalization verified.")


Final manifest status: SUCCESS
Run ID: 2308384d-2c39-41e2-816a-4bea198b310a
Batch ID: 4ebccb45-0586-4fa4-ae40-72d4a5b7c581
Source records: 25000
Written records: 25000
Rejected records: 0
Target Delta version: 3
Lease cleared: True
PASS: Manifest finalization verified.


In [0]:

# CELL 42 — FINAL END-TO-END INVENTORY (READ-ONLY)

final_bronze_df = spark.read.format("delta").load(BRONZE_PATH)

display(
    final_bronze_df
    .groupBy("_source_file")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("product_id").alias("distinct_product_ids"),
        F.sum(
            F.when(F.col("product_id").isNull(), 1).otherwise(0)
        ).alias("null_product_ids")
    )
    .orderBy("_source_file")
)

final_products_manifest = (
    spark.read.format("delta")
    .load(MANIFEST_PATH)
    .filter(F.col("dataset") == DATASET_NAME)
    .select("file_name", "status", "source_record_count",
            "written_record_count", "processed_content_hash")
    .orderBy("file_name")
)

display(final_products_manifest)

final_total = final_bronze_df.count()
assert final_total == 50000

assert (
    final_bronze_df.filter(
        F.col("_source_file") == "products_20261007.json"
    ).count() == 25000
)

assert (
    final_bronze_df.filter(
        F.col("_source_file") == "products_20261008.json"
    ).count() == 25000
)

print("PASS: Final Products Bronze inventory verified.")
print("Total Bronze records:", final_total)


_source_file,record_count,distinct_product_ids,null_product_ids
products_20261007.json,25000,25000,0
products_20261008.json,25000,25000,0


file_name,status,source_record_count,written_record_count,processed_content_hash
products_20261007.json,SUCCESS,25000,25000,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118
products_20261008.json,SUCCESS,25000,25000,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b


PASS: Final Products Bronze inventory verified.
Total Bronze records: 50000


In [0]:

# CELL 43 — ISOLATED RECOVERY TEST CONFIGURATION (NO WRITES)

TEST_DATASET = "products_recovery_test"
TEST_FILE = "products_recovery_20261007.json"

TEST_ROOT = BASE_PATH + "metadata/recovery_test/"
TEST_MANIFEST_PATH = TEST_ROOT + "file_manifest/"
TEST_BRONZE_PATH = TEST_ROOT + "bronze/products/"

print("Test dataset:", TEST_DATASET)
print("Test file:", TEST_FILE)
print("Test root:", TEST_ROOT)
print("Test manifest path:", TEST_MANIFEST_PATH)
print("Test Bronze path:", TEST_BRONZE_PATH)

assert TEST_MANIFEST_PATH != MANIFEST_PATH
assert TEST_BRONZE_PATH != BRONZE_PATH
assert TEST_DATASET != DATASET_NAME

print("PASS: Recovery test paths are isolated.")
print("No production paths will be modified by the test.")


Test dataset: products_recovery_test
Test file: products_recovery_20261007.json
Test root: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/recovery_test/
Test manifest path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/recovery_test/file_manifest/
Test Bronze path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/recovery_test/bronze/products/
PASS: Recovery test paths are isolated.
No production paths will be modified by the test.
